In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from zipfile import ZipFile

zip_path = "/content/drive/MyDrive/archive (2).zip"
extract_path = "/content/drive/MyDrive/dataset"

with ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Extraction Done ?")

In [ ]:
import os

print(os.listdir("/content/drive/MyDrive/dataset"))

In [ ]:
import os
import pandas as pd

data = []
base_path = "/content/drive/MyDrive/dataset"  # Change this to your dataset path

# Loop through all Excel files
for root, dirs, files in os.walk(base_path):
    for file in files:
        if file.endswith(".xlsx"):
            file_path = os.path.join(root, file)
            try:
                df = pd.read_excel(file_path)
                data.append(df)
            except:
                print("Error reading:", file_path)

# Combine all Excel files into one DataFrame
final_df = pd.concat(data, ignore_index=True)
print("Total rows:", final_df.shape[0])
print("Columns:", final_df.columns)

In [ ]:
# Replace NaNs with empty strings
final_df.fillna('', inplace=True)

In [ ]:
# Path to save the final CSV
csv_path = "/content/drive/MyDrive/dataset/full_dataset_all_columns.csv"

# Save to CSV
final_df.to_csv(csv_path, index=False)
print(f"? Full dataset with all columns saved to: {csv_path}")

In [ ]:
# Load CSV to check
df_check = pd.read_csv(csv_path)
print(df_check.head())
print(df_check.columns)
print("Total rows:", df_check.shape[0])

In [ ]:
import pandas as pd

# Path to your full CSV
csv_path = "/content/drive/MyDrive/dataset/full_dataset_all_columns.csv"
df = pd.read_csv(csv_path)

# Check columns
print(df.columns)

In [ ]:
import pandas as pd

# Load CSV (or Excel)
df = pd.read_csv("/content/drive/MyDrive/dataset/full_dataset_all_columns.csv")

print(df.shape)
print(df.head())

In [ ]:
df.columns = df.columns.str.strip()
print(df.columns)

In [ ]:
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/dataset/full_dataset_all_columns.csv")

In [ ]:
df.drop(columns=['Product Link', 'Image Link'], inplace=True)

In [ ]:
print(df.columns)
print(df.head())

In [ ]:
import pandas as pd
import re

df.fillna('', inplace=True)

# Clean MRP
df['MRP'] = df['MRP'].astype(str).str.replace('?', '', regex=False)
df['MRP'] = df['MRP'].str.replace(',', '', regex=False)
df['MRP'] = pd.to_numeric(df['MRP'], errors='coerce').fillna(0)

# Clean EAN
df['EAN Code'] = df['EAN Code'].astype(str).str.replace('&&', '', regex=False)

In [ ]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

df['SKU Name'] = df['SKU Name'].apply(clean_text)
df['About the Product'] = df['About the Product'].apply(clean_text)
df['Brand'] = df['Brand'].apply(clean_text)
df['SKU Size'] = df['SKU Size'].astype(str)

In [ ]:
df['text'] = (
    df['SKU Name'] + ' ' +
    df['About the Product'] + ' ' +
    df['Brand'] + ' ' +
    df['SKU Size']
)

In [ ]:
df = df[df['text'].str.strip() != '']
df.drop_duplicates(subset=['text'], inplace=True)

In [ ]:
print("Shape:", df.shape)   # rows, columns
print(df.head())            # first 5 rows
print(df.columns)           # column names

In [ ]:
print("Empty text rows:", (df['text'].str.strip() == '').sum())

In [ ]:
print("Duplicates:", df.duplicated(subset=['text']).sum())

In [ ]:
print(df.isnull().sum())

In [ ]:
df.to_csv("/content/drive/MyDrive/dataset/cleaned_dataset.csv", index=False, encoding='utf-8-sig')

In [ ]:
from google.colab import files
files.download("/content/drive/MyDrive/dataset/cleaned_dataset.csv")

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/dataset/cleaned_dataset.csv")

In [ ]:
X_text = df['text']   # input
y = df['Sub-sub-Category']   # output (target)

In [ ]:
X_train_text, X_test_text, y_train, y_test = train_test_split(
    X_text, y,
    test_size=0.2,       # 80% train, 20% test
    random_state=42
)

In [ ]:
vectorizer = TfidfVectorizer(max_features=10000)

# Fit ONLY on training data
X_train = vectorizer.fit_transform(X_train_text)

# Transform test data
X_test = vectorizer.transform(X_test_text)

In [ ]:
model = KNeighborsClassifier(n_neighbors=5, metric='cosine')
model.fit(X_train, y_train)

In [ ]:
y_pred = model.predict(X_test)

In [ ]:
print("Accuracy:", accuracy_score(y_test, y_pred))

In [ ]:
print(classification_report(y_test, y_pred))

In [ ]:
def predict_product(text):
    vec = vectorizer.transform([text])
    return model.predict(vec)[0]

print(predict_product("herbal shampoo for hair fall"))

In [ ]:
import re

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def predict_product(text):
    text = clean_text(text)   # ?? important
    vec = vectorizer.transform([text])
    return model.predict(vec)[0]

In [ ]:
print("Shape:", df.shape)
print("Columns:", df.columns)

In [ ]:
print(df.isnull().sum())

In [ ]:
df['EAN Code'] = df['EAN Code'].fillna('unknown')
df['Brand'] = df['Brand'].fillna('unknown')
df['SKU Size'] = df['SKU Size'].fillna('unknown')
df['About the Product'] = df['About the Product'].fillna('')

In [ ]:
print(df.isnull().sum())

In [ ]:
df['text'] = (
    df['SKU Name'] + ' ' +
    df['SKU Name'] + ' ' +   # boost importance
    df['About the Product'] + ' ' +
    df['Brand'] + ' ' +
    df['SKU Size']
)

In [ ]:
import re

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

df['text'] = df['text'].apply(clean_text)

In [ ]:
print(df.isnull().sum())
print("Empty text:", (df['text'].str.strip() == '').sum())

In [ ]:
df.to_csv("/content/drive/MyDrive/dataset/cleaned_dataset_final.csv", index=False, encoding='utf-8-sig')

In [ ]:
df.to_csv("/content/drive/MyDrive/dataset/cleaned_dataset_final.csv", index=False)

In [ ]:
from google.colab import files
files.download("/content/drive/MyDrive/dataset/cleaned_dataset_final.csv")

In [ ]:
print(df.shape)
print(df.isnull().sum())

In [ ]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/dataset/cleaned_dataset_final.csv")

In [ ]:
print(df.columns)


In [ ]:
X_text = df['text']
y = df['Sub-sub-Category']

print(X_text.head())
print(y.head())

In [ ]:
from sklearn.model_selection import train_test_split

X_train_text, X_test_text, y_train, y_test = train_test_split(
    X_text, y,
    test_size=0.2,
    random_state=42
)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1,2),
    stop_words='english'
)

X_train = vectorizer.fit_transform(X_train_text)
X_test = vectorizer.transform(X_test_text)

print(X_train.shape)

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

In [ ]:
from sklearn.metrics import accuracy_score

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))

In [ ]:
def predict_product(text):
    vec = vectorizer.transform([text])
    return model.predict(vec)[0]

print(predict_product("shampoo for hair fall"))
print(predict_product("face wash for oily skin"))

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

X_full = vectorizer.transform(df['text'])

In [ ]:
def recommend_products(text, top_n=5):
    vec = vectorizer.transform([text])

    similarity = cosine_similarity(vec, X_full)

    top_indices = similarity[0].argsort()[-top_n:][::-1]

    return df.iloc[top_indices][['SKU Name', 'Brand', 'MRP']]

In [ ]:
print(recommend_products("shampoo for hair fall"))

In [ ]:
import pickle

# Save model
with open("/content/drive/MyDrive/dataset/model.pkl", "wb") as f:
    pickle.dump(model, f)

# Save vectorizer
with open("/content/drive/MyDrive/dataset/vectorizer.pkl", "wb") as f:
    pickle.dump(vectorizer, f)

print("Saved successfully!")

In [ ]:
from google.colab import files

files.download("/content/drive/MyDrive/dataset/model.pkl")
files.download("/content/drive/MyDrive/dataset/vectorizer.pkl")

In [ ]:
df.to_csv("/content/final_dataset.csv", index=False)

files.download("/content/final_dataset.csv")

In [ ]:
import pickle

model = pickle.load(open("/content/drive/MyDrive/dataset/model.pkl", "rb"))
vectorizer = pickle.load(open("/content/drive/MyDrive/dataset/vectorizer.pkl", "rb"))

def predict_product(text):
    vec = vectorizer.transform([text])
    return model.predict(vec)[0]

print(predict_product("shampoo for hair fall"))

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

X_full = vectorizer.transform(df['text'])

def recommend_products(text, top_n=5):
    vec = vectorizer.transform([text])

    sim = cosine_similarity(vec, X_full)

    idx = sim[0].argsort()[-top_n:][::-1]

    return df.iloc[idx][['SKU Name', 'Brand', 'MRP']]


In [ ]:
print(recommend_products("shampoo for hair fall"))